In [3]:
import os
import glob
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# 1. Define folder path and file-to-model name mapping
# Adjust path to 'buffers/' if running notebook outside the buffers folder
buffer_dir = "buffers"  # use "." if notebook is in the same folder as csv files

model_files = {
    "gbr_random_buffer.csv": "Gradient Boosting",
    "linear_regression_random_buffer.csv": "Linear Regression",
    "m5p_random_buffer.csv": "M5P",
    "random_forest_random_buffer.csv": "Random Forest",
    "xgbr_random_buffer.csv": "XGBoost"
}

# 2. Metric calculation function
def compute_metrics(group):
    actual = group['actual']
    predicted = group['predicted']
    
    # 1. Mean Squared Error (MSE)
    mse = np.mean((actual - predicted) ** 2)
    
    # 2. Root Mean Squared Error (RMSE)
    rmse = np.sqrt(mse)
    
    # 3. Pearson Correlation (R)
    if len(actual) > 1 and np.std(actual) > 0 and np.std(predicted) > 0:
        r, _ = pearsonr(actual, predicted)
    else:
        r = np.nan
        
    # 4. Relative Absolute Error (RAE %)
    mean_actual = np.mean(actual)
    mae_model = np.sum(np.abs(actual - predicted))
    mae_naive = np.sum(np.abs(actual - mean_actual))
    rae = (mae_model / mae_naive) * 100 if mae_naive != 0 else np.nan
    
    # 5. Mean Absolute Percentage Error (MAPE %)
    valid_mask = actual != 0
    if np.any(valid_mask):
        mape = np.mean(np.abs((actual[valid_mask] - predicted[valid_mask]) / actual[valid_mask])) * 100
    else:
        mape = np.nan
    
    return pd.Series({
        'MSE': round(mse, 4),
        'RMSE': round(rmse, 4),
        'R': round(r, 4),
        'RAE (%)': round(rae, 2),
        'MAPE (%)': round(mape, 2)
    })

# 3. Loop over each CSV file and collect results
all_results = []

for file_name, model_name in model_files.items():
    # Construct full file path
    file_path = os.path.join(buffer_dir, file_name) if os.path.exists(buffer_dir) else file_name
    
    if os.path.exists(file_path):
        print(f"Processing: {model_name} ({file_name})...")
        df = pd.read_csv(file_path)
        df.columns = df.columns.str.strip()
        
        # Calculate per-commodity metrics for this model
        metrics_df = df.groupby('commodity_name').apply(compute_metrics).reset_index()
        
        # Insert Model column at the beginning
        metrics_df.insert(0, 'Model', model_name)
        
        all_results.append(metrics_df)
    else:
        print(f"Warning: File not found -> {file_path}")

# 4. Combine all model results into a single DataFrame
combined_df = pd.concat(all_results, ignore_index=True)

# Sort by commodity_name and Model for clean presentation
combined_df = combined_df.sort_values(by=['commodity_name', 'Model']).reset_index(drop=True)

# 5. Save master CSV
output_csv = "all_random_models_per_product_evaluation_metrics.csv"
combined_df.to_csv(output_csv, index=False)

print(f"\nCompleted! Consolidated metrics saved to: {output_csv}")
print("\nPreview of combined dataset:")
display(combined_df.head(10))

Processing: Gradient Boosting (gbr_random_buffer.csv)...
Processing: Linear Regression (linear_regression_random_buffer.csv)...
Processing: M5P (m5p_random_buffer.csv)...
Processing: Random Forest (random_forest_random_buffer.csv)...
Processing: XGBoost (xgbr_random_buffer.csv)...

Completed! Consolidated metrics saved to: all_random_models_per_product_evaluation_metrics.csv

Preview of combined dataset:


,Model,commodity_name,MSE,RMSE,R,RAE (%),MAPE (%)
0,Gradient Boosting,'Ata (Packet)',0.4893,0.6995,0.9745,18.63,0.80
1,Linear Regression,'Ata (Packet)',0.7320,0.8556,0.9681,25.73,1.12
2,M5P,'Ata (Packet)',0.1811,0.4255,0.9900,5.62,0.24
3,Random Forest,'Ata (Packet)',6.8446,2.6162,0.9060,88.72,3.86
4,XGBoost,'Ata (Packet)',0.2834,0.5323,0.9846,13.00,0.56
5,Gradient Boosting,'Ata (loose) - White',1.9151,1.3839,0.9027,27.84,1.57
6,Linear Regression,'Ata (loose) - White',3.7364,1.9330,0.7957,40.45,2.28
7,M5P,'Ata (loose) - White',1.7464,1.3215,0.9104,19.70,1.10
8,Random Forest,'Ata (loose) - White',9.2616,3.0433,0.5796,97.40,5.61
9,XGBoost,'Ata (loose) - White',0.8248,0.9082,0.9587,18.00,1.02


In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

df = pd.read_csv('all_random_models_per_product_evaluation_metrics.csv')

# Set style
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Mean metrics dataframe
avg_df = df.groupby('Model').agg({
    'R': 'mean',
    'RAE (%)': 'mean',
    'MAPE (%)': 'mean',
    'RMSE': 'mean'
}).reset_index()

# Sort order based on overall ranking (XGBoost, M5P, Gradient Boosting, Random Forest, Linear Regression)
model_order = ['XGBoost', 'M5P', 'Gradient Boosting', 'Random Forest', 'Linear Regression']
avg_df['Model'] = pd.Categorical(avg_df['Model'], categories=model_order, ordered=True)
avg_df = avg_df.sort_values('Model')

palette = ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728', '#9467bd']

# Plot 1: Pearson R
sns.barplot(data=avg_df, x='Model', y='R', ax=axes[0, 0], palette=palette)
axes[0, 0].set_title('Average Pearson Correlation (R) ↑ Higher is Better', fontsize=12, fontweight='bold')
axes[0, 0].set_ylim(0.6, 1.0)
for p in axes[0, 0].patches:
    axes[0, 0].annotate(f"{p.get_height():.4f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')

# Plot 2: RAE (%)
sns.barplot(data=avg_df, x='Model', y='RAE (%)', ax=axes[0, 1], palette=palette)
axes[0, 1].set_title('Average Relative Absolute Error (RAE %) ↓ Lower is Better', fontsize=12, fontweight='bold')
for p in axes[0, 1].patches:
    axes[0, 1].annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')

# Plot 3: MAPE (%)
sns.barplot(data=avg_df, x='Model', y='MAPE (%)', ax=axes[1, 0], palette=palette)
axes[1, 0].set_title('Average MAPE (%) ↓ Lower is Better', fontsize=12, fontweight='bold')
for p in axes[1, 0].patches:
    axes[1, 0].annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')

# Plot 4: RMSE
sns.barplot(data=avg_df, x='Model', y='RMSE', ax=axes[1, 1], palette=palette)
axes[1, 1].set_title('Average RMSE ↓ Lower is Better', fontsize=12, fontweight='bold')
for p in axes[1, 1].patches:
    axes[1, 1].annotate(f"{p.get_height():.2f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')

plt.tight_layout()
plt.savefig('random_model_comparison_metrics.png', dpi=300)
plt.close()
print("Chart generated successfully!")

C:\Users\Admin\AppData\Local\Temp\ipykernel_35792\1769585655.py:27: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=avg_df, x='Model', y='R', ax=axes[0, 0], palette=palette)
C:\Users\Admin\AppData\Local\Temp\ipykernel_35792\1769585655.py:35: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=avg_df, x='Model', y='RAE (%)', ax=axes[0, 1], palette=palette)
C:\Users\Admin\AppData\Local\Temp\ipykernel_35792\1769585655.py:42: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=avg_df, x='Model', y='MAPE (%)', ax=axes[1, 0], palette=palette)
C:\

Chart generated successfully!
